In [1]:
#Checking the installed Java version
!java -version
!pip install pyspark 


openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-124.04, mixed mode, sharing)


In [2]:
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless

Hit:1 https://download.docker.com/linux/ubuntu noble InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-security InRelease 
Hit:7 https://packages.cloud.google.com/apt cloud-sdk InRelease                
Hit:8 https://security.ubuntu.com/ubuntu noble-security InRelease              
Hit:9 http://deb.wakemeops.com/wakemeops stable InRelease           
Hit:10 https://archive.ubuntu.com/ubuntu noble InRelease
Hit:11 https://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:12 https://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk-he

In [3]:
# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local[*]")\
        .appName("ML") \
        .getOrCreate()
print("Spark ready:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/14 21:09:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark ready: 3.5.0


In [4]:
# Load the dataset into a Spark DataFrame
file_location       = "/teamspace/studios/this_studio/Projeto/Airline_Delay_Cause.csv" # location of the file in Data>dbfs

# Import data
airline_delays = spark.read.load(file_location,
                     format      = "csv",           
                     sep         = ",",           
                     inferSchema = "true",        
                     header      = "true")

In [5]:
#RDDs (Low-Level API) 
# Dataset: Airline_Delay_Cause.csv

from pyspark import SparkContext
sc = SparkContext.getOrCreate()

#First: data ingestion and quality check
# We read the data to understand the schema.
# "inferSchema" causes a pass over the data (What is a string, intenger or double).In production Big Data, we would manually define the schema to save time, but for exploration, this is acceptable.

df_initial = airline_delays

print("Schema Overview")
df_initial.printSchema()

print("Total Row Count")
# .count() is an action that triggers a job, but it is optimized to run efficiently across nodes.Just now Spark processes the "plan"
print(f"Total flights in dataset: {df_initial.count()}")

Schema Overview
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- carrier_name: string (nullable = true)
 |-- airport: string (nullable = true)
 |-- airport_name: string (nullable = true)
 |-- arr_flights: double (nullable = true)
 |-- arr_del15: double (nullable = true)
 |-- carrier_ct: double (nullable = true)
 |-- weather_ct: double (nullable = true)
 |-- nas_ct: double (nullable = true)
 |-- security_ct: double (nullable = true)
 |-- late_aircraft_ct: double (nullable = true)
 |-- arr_cancelled: double (nullable = true)
 |-- arr_diverted: double (nullable = true)
 |-- arr_delay: double (nullable = true)
 |-- carrier_delay: double (nullable = true)
 |-- weather_delay: double (nullable = true)
 |-- nas_delay: double (nullable = true)
 |-- security_delay: double (nullable = true)
 |-- late_aircraft_delay: double (nullable = true)

Total Row Count
Total flights in dataset: 171666


### Big data safety (note): Why using .take(n) or .show() instead of dataset.collect()?

While collect() retrieves all data from the distributed worker nodes and bring it into the driver´s single-node memory (which is a problem when we work on datasets of Terabytes, because the driver would crash immediately (out of memory error)
.take(n) or .show() only retieve a small subset of the final aggragated results to the driver, leaving the heavy data on the cluster.

In [6]:
# RDD (Resilient Distributed Dataset) Approach
# In order to demonstrate parsing, mapping, reducing by key, and sorting safely, our goalis to Calculate total delay minutes per Carrier using MapReduce

# 1. Loading text file
# 'sc' is the SparkContext (the connection to the cluster)
sc = spark.sparkContext
lines_rdd = sc.textFile("Airline_Delay_Cause.csv")

# 2. Separating Header from Data
# In RDDs, we have to manually handle the header row

header = lines_rdd.first()
# Separate  first line (Header) from Data

data_rdd = lines_rdd.filter(lambda row: row != header)
#creates new RDD woth filtred header (first line)

# 3. Defining Mapper function - transforming a text line into a pair key-value
# This function runs on every single line of our CSV
def extract_carrier_delay(line):
    try:
        parts = line.split(',')
        # Split the CSV line by comma

        # SAFETY: Check if we have enough columns to avoid IndexErrors
        if len(parts) < 16:
            return ("Invalid", 0.0)
        #This way, we still continue processing the other data instead of stopping because of ex:a bad registry.

        # Based on the standard structure of this dataset:
        # Index 3 is 'carrier_name' (e.g., "American Airlines")
        # Index 16 is 'carrier_delay' (Minutes)
        carrier = parts[3]
        delay_str = parts[16]

        # Handle empty strings (assuming time is 0) or conversion errors (convert delay string to a decimal number/float)
        delay = float(delay_str) if delay_str and delay_str != '' else 0.0

        return (carrier, delay)
     # If a row is malformed, we return a dummy value rather than crashing the whole job
    except Exception as e:
        return ("Error_Row", 0.0)
   
        

# 4. Executing MapReduce
# MAP: Convert lines to (Carrier, Delay) pairs
mapped_rdd = data_rdd.map(extract_carrier_delay)

# REDUCE: Agreagates all delays to same key (carrier) and sum them 
# reduceByKey is "Big Data Safe" because it sums on the local nodes first
total_delays = mapped_rdd.reduceByKey(lambda a, b: a + b)

# 5. Output Results - sorting by delay (descending) and taking the top 5
# NOTE: We use take(5), NOT collect(). Because it´s dangerous for Big Data (as we already said before).
top_5_carriers = total_delays.sortBy(lambda x: x[1], ascending=False).take(5)

print("Top 5 Carriers by Carrier Delay (RDD Method)")
for carrier, minutes in top_5_carriers:
    
    # Excluding our error/invalid keys if they appear
    if carrier not in ["Invalid", "Error_Row"]:
        print(f"{carrier}: {minutes:,.2f} Minutes")

Top 5 Carriers by Carrier Delay (RDD Method)
Southwest Airlines Co.: 124,294,977.00 Minutes
American Airlines Inc.: 103,761,183.00 Minutes
SkyWest Airlines Inc.: 87,077,507.00 Minutes
Delta Air Lines Inc.: 79,047,428.00 Minutes
United Air Lines Inc.: 70,288,103.00 Minutes


In [7]:
# Part 2: DataFrames & SparkSQL
# Goal: Analyze delay patterns using High-Level APIs.
# Unlike RDDs, DataFrames use the 'Catalyst Optimizer' to optimize query execution plans automatically.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Loadind and inspecting schema
# inferSchema=True triggers a read job. In a real PB-scale scenario, we would define schema manually to save time.
df = spark.read.csv("Airline_Delay_Cause.csv", header=True, inferSchema=True)

# Quick check of column names to ensure we are using the right ones
# (This helps debug if column names differ in your specific CSV version)
print("Columns found:", df.columns)

# 2. Data cleaning
# Casting columns to proper types (Double) and filling Nulls with 0.
# We select 'arr_delay' (Arrival Delay) and 'carrier' as our main focus.
df_clean = df.withColumn("arr_delay", F.col("arr_delay").cast("double")) \
             .fillna(0, subset=["arr_delay"])

# 3. Insight A: The "General" View (Using Spark SQL)
# Which airports have the highest average delays?
df_clean.createOrReplaceTempView("flights")

query_airport = """
    SELECT
        airport,
        ROUND(AVG(arr_delay), 2) as avg_delay_min
    FROM flights
    GROUP BY airport
    ORDER BY avg_delay_min DESC
    LIMIT 5
"""

print("\n Insight A: Top 5 Airports by Average Delay (SQL) ")
spark.sql(query_airport).show()

# 4. Insight B: The "Advanced" View (Using Window Functions)
# Technical Skill: Window Functions allow us to look at a row in the context of its group.
# Question: Which specific flights were anomalies compared to their airport's average?

# Defining the Window: We partition by Airport.
# Big data safety (note): Partitioning by key is safe. An empty partitionBy() would move all data to one node, crashing a Big Data job.
windowSpec = Window.partitionBy("airport")

# Calculating the average delay for THAT airport and adding it to every row
df_enhanced = df_clean.withColumn("airport_avg_delay", F.avg("arr_delay").over(windowSpec))

# Calculating the "Deviation": How much worse was this flight than the airport average?
df_final = df_enhanced.withColumn("deviation_from_avg", F.col("arr_delay") - F.col("airport_avg_delay"))

print("\n Insight B: Extreme Outliers (Flights much worse than their airport norm) ")
# We use select() to keep the output clean and readable
df_final.select("carrier", "airport", "arr_delay", "airport_avg_delay", "deviation_from_avg") \
        .orderBy(F.desc("deviation_from_avg")) \
        .show(5)

Columns found: ['year', 'month', 'carrier', 'carrier_name', 'airport', 'airport_name', 'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted', 'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']

 Insight A: Top 5 Airports by Average Delay (SQL) 


+-------+-------------+
|airport|avg_delay_min|
+-------+-------------+
|    ORD|     28688.19|
|    DFW|     26513.34|
|    ATL|     23149.15|
|    DEN|     21660.03|
|    SFO|     19627.08|
+-------+-------------+


 Insight B: Extreme Outliers (Flights much worse than their airport norm) 
+-------+-------+---------+------------------+------------------+
|carrier|airport|arr_delay| airport_avg_delay|deviation_from_avg|
+-------+-------+---------+------------------+------------------+
|     AA|    DFW| 438783.0|26513.337217771303|412269.66278222867|
|     DL|    ATL| 429194.0| 23149.15165562914|406044.84834437084|
|     AA|    DFW| 416577.0|26513.337217771303|390063.66278222867|
|     DL|    ATL| 395609.0| 23149.15165562914|372459.84834437084|
|     AA|    DFW| 376057.0|26513.337217771303|349543.66278222867|
+-------+-------+---------+------------------+------------------+
only showing top 5 rows



### Analysis Part 2: DataFrames & High-Level APIs

In this section, we transition from RDDs to Spark DataFrames. Why?

DataFrames utilize Spark's Catalyst Optimizer. Unlike RDDs, where the developer manually defines the execution steps (Map -> Reduce), DataFrames allow Spark to analyze the logical plan and optimize it (e.g., pushing down filters to the data source) before execution.

Advanced Technique: We implement a Window Function in the second part of this block. This allows us to calculate relative performance (e.g., how did this flight compare to the average of its specific airport?) without performing expensive self-joins.

Big Data safety (note): We continue to use .show() and .limit() to ensure we never pull the full dataset into the driver's memory.